In [0]:
import requests
from pyspark.sql import functions as F

years = ["2020", "2021", "2022", "2023", "2024", "2025"]
master_df = None

print("Starting pipeline")

# Fetching and processing data for each year
for year in years:
    print(f"Fetching Weather Data for {year}...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 52.4064,
        "longitude": 16.9252,
        "start_date": f"{year}-01-01", 
        "end_date": f"{year}-12-31",
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "timezone": "Europe/Warsaw"
    }

    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        print('Parsing responponse...')

        chunk_raw_df = spark.createDataFrame([data]) 
        
        # Explode the hourly data into individual rows
        chunk_exploded = chunk_raw_df.select(
            F.col("latitude"),
            F.col("longitude"),
            F.explode(
                F.arrays_zip(
                    "hourly.time",
                    "hourly.temperature_2m",
                    "hourly.relative_humidity_2m",
                    "hourly.wind_speed_10m"
                )
            ).alias("sensor_readings")
        )
        # Select sensor readings as individual columns
        chunk_clean = chunk_exploded.select(
            F.col("latitude"),
            F.col("longitude"),
            F.col("sensor_readings.time").cast("timestamp").alias("timestamp"),
            F.col("sensor_readings.temperature_2m").alias("temperature_celsius"),
            F.col("sensor_readings.relative_humidity_2m").alias("humidity_percent"),
            F.col("sensor_readings.wind_speed_10m").alias("wind_speed_kmh")
        )
        
        # Append to master DataFrame
        if master_df is None:
            master_df = chunk_clean 
        else:
            master_df = master_df.union(chunk_clean) 
            
    else:
        print(f"Failed on {year}. Status Code: {response.status_code}")

print(f"Pipeline complete! Total rows ingested: {master_df.count()}")
display(master_df)

In [0]:
from pyspark.sql.window import Window

# Anomaly Detection: Flag temperature changes greater than 3°C within an hour

gold_df = master_df.withColumn("temperature_celsius", F.col("temperature_celsius").cast("double"))

window_spec = Window.partitionBy("latitude", "longitude").orderBy("timestamp")

gold_df = gold_df.withColumn("prev_temp", F.lag("temperature_celsius").over(window_spec))

gold_df = gold_df.withColumn(
    "temp_change", 
    F.abs(F.col("temperature_celsius") - F.col("prev_temp"))
).withColumn(
    "is_anomaly",
    F.when(F.col("temp_change") > 3.0, True).otherwise(False)
)

In [0]:
full_weather_data = gold_df.withColumn(
    "temp_change", F.round("temp_change", 2) 
)
full_weather_data.createOrReplaceTempView("full_weather_data_temp")

In [ ]:
%sql

CREATE TABLE IF NOT EXISTS full_weather_data (
    latitude DOUBLE,
    longitude DOUBLE,
    timestamp TIMESTAMP,
    temperature_celsius DOUBLE,
    humidity_percent DOUBLE,
    wind_speed_kmh DOUBLE,
    prev_temp DOUBLE,
    temp_change DOUBLE,
    is_anomaly BOOLEAN
) USING DELTA;

INSERT OVERWRITE full_weather_data
SELECT 
    latitude,
    longitude,
    timestamp,
    temperature_celsius,
    humidity_percent,
    wind_speed_kmh,
    prev_temp,
    temp_change,
    is_anomaly
FROM 
    full_weather_data_temp;

CREATE OR REPLACE VIEW weather_gold_anomalies_view AS
SELECT 
    timestamp,
    temperature_celsius,
    prev_temp,
    temp_change
FROM 
    full_weather_data
WHERE 
    is_anomaly = TRUE;

SELECT * FROM weather_gold_anomalies_view ORDER BY temp_change DESC;